In [3]:
"""The code is originally from https://nb.bohrium.dp.tech/detail/3413343451

1. the use of vectroize(u) is removed, it is not necessary for the code to work, also it is slow
"""

'The code is originally from https://nb.bohrium.dp.tech/detail/3413343451\n\n1. the use of vectroize(u) is removed, it is not necessary for the code to work, also it is slow\n'

### problem 1 settings 

In [31]:
import numpy as np
import math
import time
from numpy.linalg import solve
import matplotlib.pyplot as plt
from scipy.linalg import lstsq
import torch 
if torch.cuda.is_available():  
    device = "cuda" 
else:  
    device = "cpu" 
def PiecewiseGQ1D_weights_points(x_l,x_r,Nx, order):
    """ Output the coeffients and weights for piecewise Gauss Quadrature 
    Parameters
    ----------
    x_l : float 
    left endpoint of an interval 
    
    x_r: float
    right endpoint of an interval 
    
    integration_intervals: int
    number of subintervals for integration
    
    Returns
    -------
    coef1_expand
    
    gw_expand
    
    integration_points
    """
    x,w = np.polynomial.legendre.leggauss(order)
    gx = torch.tensor(x).to(device)
    gx = gx.view(1,-1) # row vector 
    gw = torch.tensor(w).to(device)    
    gw = gw.view(-1,1) # Column vector 
    nodes = torch.linspace(x_l,x_r,Nx+1).view(-1,1).to(device) 
    coef1 = ((nodes[1:,:] - nodes[:-1,:])/2) # n by 1  
    coef2 = ((nodes[1:,:] + nodes[:-1,:])/2) # n by 1  
    coef2_expand = coef2.expand(-1,gx.size(1)) # Expand to n by p shape, -1: keep the first dimension n , expand the 2nd dim (columns)
    integration_points = coef1@gx + coef2_expand
    integration_points = integration_points.flatten().view(-1,1) # Make it a column vector
    gw_expand = torch.tile(gw,(Nx,1)) # rows: n copies of current tensor, columns: 1 copy, no change
    # Modify coef1 to be compatible with func_values
    coef1_expand = coef1.expand(coef1.size(0),gx.size(1))    
    coef1_expand = coef1_expand.flatten().view(-1,1)

    return coef1_expand.to(device)*gw_expand.to(device), integration_points.to(device)


## test 1. 

vanal_f = np.vectorize(f) is not faster. 

In [33]:
def error_plot(multi_Errors):
    plt.figure(figsize=[7, 5])
    plt.tick_params(labelsize=10)
    font2 = {
    'weight' : 'normal',
    'size'   : 22,
    }
    plt.xlabel('Degrees of freedom',font2)
    plt.ylabel('$L_2$ absolute error',font2)
    plt.xscale('log')
    plt.yscale('log')
    # Label = ['FDM','PINN','RFM']
    Label = ['RFM']
    for i in range(len(multi_Errors)):
        Error = multi_Errors[i]
        plt.plot(Error[:,0], Error[:,1], \
                 lw=1.5, ls='-', clip_on=False,\
                 marker='o', markersize=10, \
                 label = Label[i],\
                 markerfacecolor='none',\
                 markeredgewidth=1.5)
    plt.legend()
    plt.title("Comparison of accuracy on 1D Helmholtz equation")
    plt.show()

def time_plot(multi_Errors):
    plt.figure(figsize=[7, 5])
    plt.tick_params(labelsize=10)
    font2 = {
    'weight' : 'normal',
    'size'   : 22,
    }
    plt.xlabel('Degrees of freedom',font2)
    plt.ylabel('Solving time',font2)
    Label = ['FDM','PINN','RFM']
    for i in range(len(multi_Errors)):
        Error = multi_Errors[i]
        plt.plot(Error[:,0], Error[:,2], \
                 lw=1.5, ls='-', clip_on=False,\
                 marker='o', markersize=10, \
                 label = Label[i],\
                 markerfacecolor='none',\
                 markeredgewidth=1.5)
    plt.legend()
    plt.title("Comparison of efficiency on 1D Helmholtz equation")
    plt.show()
    

## RFM

In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
torch.set_default_dtype(torch.float64)

R_m = 2.0 
## original tests 
def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        if activation == 'tanh':
            nn.init.uniform_(m.weight, a = -R_m, b = R_m)
            nn.init.uniform_(m.bias, a = -R_m, b = R_m)
        if activation == 'relu':
            nn.init.uniform_(m.weight, a = 1, b = 1)
            nn.init.uniform_(m.bias, a = -1 , b = 1.2)
            nodes = -m.bias.data.squeeze()/m.weight.data.squeeze()
            if len(nodes[nodes < -1]) == 0: 
                print("all nodes are within the interval")
                m.bias.data[0] = 1.2 


# network definition
class Network(nn.Module):
    def __init__(self, d, M):
        super(Network, self).__init__()
        self.fc_layer = nn.Sequential(nn.Linear(d, M, bias=True),nn.Tanh())
        self.output_layer = nn.Linear(M, 1, bias = False)
        
    def forward(self, x):
        h = self.fc_layer(x)
        out = self.output_layer(h)
        return out

### 

explanations: 
1. $\sigma(\omega \cdot \tilde{x} +b )$ for each subdomain, $\tilde{x}$ is properly scaled 
2. store the models on each subdomain in a list 

In [35]:
# computational domain
X_min = 0.0
X_max = 8.0

# # random initialization for parameters
# def weights_init(m):
#     if isinstance(m, (nn.Conv2d, nn.Linear)):
#         nn.init.uniform_(m.weight, a = -R_m, b = R_m)
#         nn.init.uniform_(m.bias, a = -R_m, b = R_m)

class ReLUk(nn.Module):
    def __init__(self, k):
        super(ReLUk, self).__init__()
        self.k = k

    def forward(self, x):
        return torch.relu(x)**self.k 
    
class Gaussian(nn.Module):
    def __init__(self):
        super(Gaussian, self).__init__()

    def forward(self, x):
        return torch.exp(-x**2)
    
class cosine_activation(nn.Module): 
    def __init__(self):
        super(cosine_activation, self).__init__()
    def forward(self, x):
        return torch.cos(x)
    
class RFM_rep_a(nn.Module): # assume defined on [-1,1], with proper scaling, it can be used for any interval 
    def __init__(self, d, J_n, x_min, x_max, activation = 'tanh', k = 1 ):
        super(RFM_rep_a, self).__init__()
        self.d = d
        self.J_n = J_n
        self.r = (x_max - x_min) / 2.0

        self.x_c = (x_max + x_min)/2
        self.activation = activation 
        self.k = k 
        if self.activation == 'tanh':
            self.phi = nn.Sequential(nn.Linear(self.d, self.J_n, bias=True),nn.Tanh())
        if self.activation == 'relu':
            self.phi = nn.Sequential(nn.Linear(self.d, self.J_n, bias=True),ReLUk(self.k))
        if self.activation == 'sigmoid':
            self.phi = nn.Sequential(nn.Linear(self.d, self.J_n, bias=True),nn.Sigmoid())
        if self.activation == 'gaussian':
            self.phi = nn.Sequential(nn.Linear(self.d, self.J_n, bias=True),Gaussian()) 
        if self.activation == 'cosine': 
            self.phi = nn.Sequential(nn.Linear(self.d, self.J_n, bias=True),cosine_activation()) 

    def forward(self,x):
        x = (x - self.x_c) / self.r # this normalizatoin is important. allows a same random distribution to be used for all regions 
        # print(self.r)
        x = self.phi(x)
        return x


# random feature basis when using \psi^{b} as PoU function
class RFM_rep_b(nn.Module):
    def __init__(self, d, J_n, x_max, x_min):
        super(RFM_rep_b, self).__init__()
        self.d = d
        self.J_n = J_n
        self.n = x_min/(X_max-X_min) * M_p
        self.r = (x_max - x_min) / 2.0
        self.x_0 = (x_max + x_min)/2
        self.phi = nn.Sequential(nn.Linear(self.d, self.J_n, bias=True),nn.Tanh())

    def forward(self,x):
        d = (x - self.x_0) / self.r
        psi = ((d <= -3/4) & (d > -5/4)) * (1+torch.sin(2*np.pi*d))/2 + ((d <= 3/4) & (d > -3/4)) * 1.0 + ((d <= 5/4) & (d > 3/4)) * (1-torch.sin(2*np.pi*d))/2
        y = self.phi(d)
        if self.n == 0:
            psi = ((d <= 3/4) & (d > -1)) * 1.0 + ((d <= 5/4) & (d > 3/4)) * (1-torch.sin(2*np.pi*d))/2
        elif self.n == M_p-1:
            psi = ((d <= -3/4) & (d > -5/4)) * (1+torch.sin(2*np.pi*d))/2 + ((d <= 1) & (d > -3/4)) * 1.0
        else:
            psi = ((d <= -3/4) & (d > -5/4)) * (1+torch.sin(2*np.pi*d))/2 + ((d <= 3/4) & (d > -3/4)) * 1.0 + ((d <= 5/4) & (d > 3/4)) * (1-torch.sin(2*np.pi*d))/2
        return(psi*y)

# predefine the random feature functions in each PoU region
def pre_define(M_p,J_n,activation,k):
    models = []
    for n in range(M_p):
        x_min = (X_max-X_min)/M_p * n + X_min
        x_max = (X_max-X_min)/M_p * (n+1) + X_min
        model = RFM_rep_a(d = 1, J_n = J_n, x_min = x_min, x_max = x_max, activation=activation, k = k)
        model = model.apply(weights_init)
        # print(model.phi[0].weight)
        # print(model.phi[0].bias)
        # nodes = -model.phi[0].bias.data.squeeze()/model.phi[0].weight.data.squeeze()
        # print(nodes)
        model = model.double()
        # x = torch.tensor([0.1])
        # print(model(x)) 
        for param in model.parameters():
            param.requires_grad = False
        models.append(model)
    return(models)

In [52]:
# calculate the l^{infty}-norm and l^{2}-norm error for u
def test(models,M_p,J_n,Q,w,plot = False, print_res = True):
    epsilon = []
    true_values = []
    numerical_values = []
    test_Q = 2*Q
    for n in range(M_p):
        points = torch.tensor(np.linspace((X_max-X_min)/M_p * n + X_min, (X_max-X_min)/M_p * (n+1) + X_min, test_Q+1),requires_grad=False).reshape([-1,1])
        out = models[n](points)
        values = out.detach().numpy()
        numerical_value = np.dot(values, w[n*J_n:(n+1)*J_n,:]) 
        true_value = u(points.numpy()).reshape([-1,1])
        numerical_values.extend(numerical_value) # append
        true_values.extend(true_value)
        epsilon.extend(true_value - numerical_value)
    true_values = np.array(true_values)
    numerical_values = np.array(numerical_values)
    epsilon = np.array(epsilon)
    epsilon = np.maximum(epsilon, -epsilon)
    if print_res: 
        print('R_m=%s,M_p=%s,J_n=%s,Q=%s'%(R_m,M_p,J_n,Q))
        print('L_infty error =',epsilon.max(),', L_2 error =',math.sqrt(8*sum(epsilon*epsilon)/len(epsilon)))
    x = [((X_max - X_min)/M_p)*i / test_Q  for i in range(M_p*(test_Q+1))]
    return(math.sqrt((X_max-X_min)*sum(epsilon*epsilon)/len(epsilon)))



 modify the previous code to solve an l^2 minimization problem 

Collocation points. $M_p$ subdomains 
1. interior points: $M_p \times Q$
2. boundary points: 2 
3. smoothness conditions: $(M_p - 1 ) \times 2$
Design matrix is of size: # of data points $\times $ # of dofs (features) 

Each subdomain corresponds to a model. All models are stored in a list. 

model: data points $\to $ features (column vector of basis function values)

Main function:
1. points are pre-allocated in list 

In [ ]:
# Assembling the matrix A,f in linear system 'Au=f'
def assemble_matrix(models,points,M_p,J_n,Q,lamb):
    """This is the data matrix. Each row of A is a data point constraint. 
    """
    A_I = np.zeros([M_p*Q, M_p*J_n]) # PDE term
    A_B = np.zeros([2, M_p*J_n]) # boundary condition
    A_C_0 = np.zeros([M_p-1, M_p*J_n]) # 0-order smoothness condition
    A_C_1 = np.zeros([M_p-1, M_p*J_n]) # 1-order smoothness condition
    # f = np.zeros([M_p*Q + 2*(M_p - 1) + 2, 1])
    f = np.zeros([M_p*Q + (M_p - 1) + 2, 1])
    
    
    for n in range(M_p):
        # forward and grad
        point = torch.tensor(points[n], requires_grad=True)
        out = models[n](point)
        # if n == 0:
            # print("output")
            # print(out)
        values = out.detach().numpy()
        value_l, value_r = values[0,:], values[-1,:]
        grad1 = []
        grad2 = []
        for i in range(J_n):
            g1 = torch.autograd.grad(outputs=out[:,i], inputs=point,
                                  grad_outputs=torch.ones_like(out[:,i]),
                                  create_graph = True, retain_graph = True)[0]
            grad1.append(g1.squeeze().detach().numpy())
            
            g2 = torch.autograd.grad(outputs=g1[:,0], inputs=point,
                                  grad_outputs=torch.ones_like(out[:,i]),
                                  create_graph = False, retain_graph = True)[0]
            grad2.append(g2.squeeze().detach().numpy())
        grad1 = np.array(grad1).T
        grad2 = np.array(grad2).T
        grad_l = grad1[0,:]
        grad_r = grad1[-1,:]

        # Lu = - grad2 + lamb * values # -laplacian + \lambda u 
        Lu = values 
        
        # Lu = f condition
        A_I[n*Q:(n + 1)*Q, n*J_n:(n + 1)*J_n] = Lu[:Q,:]
        f[n*Q:(n + 1)*Q, :] = F(points[n], lamb).reshape([-1,1])[:Q] #rhs function or target function 
        
        # boundary conditions
        if n == 0:
            A_B[0, :J_n] = value_l 
        if n == M_p-1:
            A_B[1, -J_n:] = value_r
        
        # smoothness conditions
        if M_p > 1:
            if n == 0 :
                A_C_0[0, :J_n] = -value_r
                # A_C_1[0, :J_n] = -grad_r
            elif n == M_p - 1:
                A_C_0[M_p - 2, -J_n:] = value_l
                # A_C_1[M_p - 2, -J_n:] = grad_l
            else:
                A_C_0[n-1,n*J_n:(n + 1)*J_n] = value_l
                # A_C_1[n-1,n*J_n:(n + 1)*J_n] = grad_l
                A_C_0[n,n*J_n:(n + 1)*J_n] = -value_r
                # A_C_1[n,n*J_n:(n + 1)*J_n] = -grad_r
    if M_p > 1:
        # A = np.concatenate((A_I,A_B,A_C_0,A_C_1),axis=0)
        A = np.concatenate((A_I,A_B,A_C_0),axis=0)
    else:
        A = np.concatenate((A_I,A_B),axis=0)
    
    # boundary conditions
    f[M_p*Q,:] = u(0.)
    f[M_p*Q+1,:] = u(8.)
    return(A,f)

def main(M_p, J_n, Q, lamb, activation, k, print_res = True):
    # prepare collocation points
    time_begin = time.time()
    points = []
    for n in range(M_p):
        x_min = (X_max-X_min)/M_p * n + X_min
        x_max = (X_max-X_min)/M_p * (n+1) + X_min
        points.append(np.linspace(x_min, x_max, Q+1).reshape([-1,1]))
    # prepare models
    models = pre_define(M_p,J_n,activation, k)
    # print(models)
    # model = models[0]
    # print( - model.phi[0].bias.data.squeeze()/model.phi[0].weight.data.squeeze())    

    # matrix define (Au=f)
    A,f = assemble_matrix(models, points, M_p, J_n, Q, lamb)
    # print("matrix")
    # print(A)
    if print_res:
        print('***********************')
        print('Matrix shape: N=%s,M=%s'%(A.shape))

    # rescaling
    c = 100.0
    for i in range(len(A)):
        ## change 
        max_a = abs(A[i,:]).max()
        max_b = A[i,:].max()
        if max_a != max_b: 
            ratio = -c/max_a
            A[i,:] = A[i,:]*ratio
            f[i] = f[i]*ratio
        else: 
            ratio = c/max_a
            A[i,:] = A[i,:]*ratio
            f[i] = f[i]*ratio
    
    # solve
    w = lstsq(A,f)[0]
    
    # test
    error = test(models,M_p,J_n,Q,w, print_res = print_res)
    
    time_end = time.time()
    return models,w, error, time_end - time_begin 

### original parameters

In [44]:


def evaluate_model(models,M_p,J_n,w,x_coord,plot = False, print_res = True):
    """
    Need M_p, J_n, w to evaluate the model on x_coord 
    """
    numerical_values = []
    for n in range(M_p):
        # points = torch.tensor(np.linspace((X_max-X_min)/M_p * n + X_min, (X_max-X_min)/M_p * (n+1) + X_min, test_Q+1),requires_grad=False).reshape([-1,1])
        x_min = (X_max-X_min)/M_p * n + X_min
        x_max = (X_max-X_min)/M_p * (n+1) + X_min
        if n==0:
            points = torch.tensor(x_coord[(x_coord >= x_min) & (x_coord <= x_max)], requires_grad=False).reshape([-1,1]) 
        else: 
            points = torch.tensor(x_coord[(x_coord > x_min) & (x_coord <= x_max)], requires_grad=False).reshape([-1,1])
        out = models[n](points)
        values = out.detach().numpy()
        numerical_value = np.dot(values, w[n*J_n:(n+1)*J_n,:]) 
        true_value = u(points.numpy()).reshape([-1,1])
        numerical_values.append(numerical_value) # append
    numerical_values = np.array(numerical_values)
    numerical_values = np.concatenate(numerical_values, axis = 0) 
    return numerical_values 


## original tests 
def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        if activation != 'relu': # Not ReLU 
            nn.init.uniform_(m.weight, a = -R_m, b = R_m)
            nn.init.uniform_(m.bias, a = -R_m, b = R_m)
        if activation == 'relu':
            nn.init.uniform_(m.weight, a = 1, b = 1)
            nn.init.uniform_(m.bias, a = -1 , b = 1.2)
            nodes = -m.bias.data.squeeze()/m.weight.data.squeeze()
            if len(nodes[nodes < -1]) == 0: 
                print("all nodes are within the interval")
                m.bias.data[0] = 1.2 

In [47]:
def output_results_Rs_latex(err_list_Rs, R_m_list, num_diff_features_list):
    num_rows = len(num_diff_features_list)
    num_cols = len(R_m_list)

    # Start LaTeX table
    latex_str = "\\begin{tabular}{c|" + "c" * num_cols + "}\n"
    
    # Header row: R values
    latex_str += "\#Features $\\backslash$ R"
    for R in R_m_list:
        latex_str += f" & {R}"
    latex_str += " \\\\\n\\hline\n"

    # Data rows: each row is for a fixed number of features
    for i, n_feat in enumerate(num_diff_features_list):
        latex_str += f"{n_feat}"
        for j in range(num_cols):
            err = err_list_Rs[j][i]  # error for R = R_m_list[j], features = num_diff_features_list[i]
            latex_str += f" & {err:.2e}"
        latex_str += " \\\\\n"

    latex_str += "\\end{tabular}"
    return latex_str

import numpy as np

R_m_list = [1, 2, 3]
num_diff_features_list = [10, 20, 30]
err_list_Rs_test = [
    np.array([1e-2, 5e-3, 1e-3]),   # errors for R = 1
    np.array([8e-3, 4e-3, 9e-4]),   # errors for R = 2
    np.array([6e-3, 3e-3, 7e-4])    # errors for R = 3
]

print(output_results_Rs_latex(err_list_Rs_test, R_m_list, num_diff_features_list))


\begin{tabular}{c|ccc}
\#Features $\backslash$ R & 1 & 2 & 3 \\
\hline
10 & 1.00e-02 & 8.00e-03 & 6.00e-03 \\
20 & 5.00e-03 & 4.00e-03 & 3.00e-03 \\
30 & 1.00e-03 & 9.00e-04 & 7.00e-04 \\
\end{tabular}


In [59]:

## L^2 minimization 

# analytical solution parameters
AA = 1
aa = 1.0*np.pi
lamb = 4
activation = 'tanh'
k = 3 # if using tanh, k is not used 

def u(x):
    return AA *  np.sin(aa * x)

# def d2u_dx2(x):
#     return -AA*(aa*aa) * np.cos(aa*(x+0.05)) 

def F(points, lamb):
    # return(- d2u_dx2(points)  + lamb * u(points)) # change 
    return u(points)


if __name__ == '__main__':
    lamb = 4
    PoU_nums = 5 
    activation = 'tanh'
    k = 2 # if using tanh, k is not used 
    num_trials = 2 
    err_list_Rs = [] 
    R_m_list = [0.75, 1.5,3, 6,9,12] 
    for R_m in R_m_list: # 0.5,1,2 
        print() 
        J_n = 50 # the number of basis functions per PoU region
        Q = 50 # the number of collocation pointss per PoU regio
        RFM_Error = np.zeros([PoU_nums,3])
        err_list = np.zeros([PoU_nums,num_trials])
        for i in range(PoU_nums): # the number of PoU regions
            for trial in range(num_trials): 
                M_p =  2 * (2**i)
                RFM_Error[i,0] = int(M_p * J_n)
                models,w, RFM_Error[i,1], RFM_Error[i,2] = main(M_p,J_n,Q,lamb,activation, k, print_res = False)
                err_list[i,trial] = RFM_Error[i,1] 
        print(err_list.mean(axis = 1)) 
        err_list_Rs.append(err_list.mean(axis = 1)) 
        
    num_diff_features_list = [J_n * 2 * (2**i) for i in range(PoU_nums)]

    print(output_results_Rs_latex(err_list_Rs, R_m_list, num_diff_features_list))

/var/folders/45/p1_q2drx75983lk7ph_1gx6mhpr5_w/T/ipykernel_8943/4274200674.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return(math.sqrt((X_max-X_min)*sum(epsilon*epsilon)/len(epsilon)))


[7.46318180e-07 5.34584248e-10 5.23820074e-12 2.50793000e-13
 7.74838310e-14]

[4.67089363e-10 1.22135729e-11 6.80556698e-13 9.33410986e-14
 3.29084530e-14]

[3.02168992e-10 1.11631961e-11 2.14469308e-12 2.27852945e-13
 7.59275754e-14]

[1.39617860e-05 4.54379011e-08 1.73663306e-09 5.28577574e-09
 2.33412126e-09]

[2.25595388e-04 1.95582660e-05 6.89047557e-05 5.75670070e-06
 2.52320743e-06]

[0.01715286 0.01242889 0.0001904  0.0005917  0.00039008]
\begin{tabular}{c|cccccc}
\#Features $\backslash$ R & 0.75 & 1.5 & 3 & 6 & 9 & 12 \\
\hline
100 & 7.46e-07 & 4.67e-10 & 3.02e-10 & 1.40e-05 & 2.26e-04 & 1.72e-02 \\
200 & 5.35e-10 & 1.22e-11 & 1.12e-11 & 4.54e-08 & 1.96e-05 & 1.24e-02 \\
400 & 5.24e-12 & 6.81e-13 & 2.14e-12 & 1.74e-09 & 6.89e-05 & 1.90e-04 \\
800 & 2.51e-13 & 9.33e-14 & 2.28e-13 & 5.29e-09 & 5.76e-06 & 5.92e-04 \\
1600 & 7.75e-14 & 3.29e-14 & 7.59e-14 & 2.33e-09 & 2.52e-06 & 3.90e-04 \\
\end{tabular}


In [60]:

## L^2 minimization 

# analytical solution parameters
AA = 1
aa = 16.0*np.pi
lamb = 4
activation = 'tanh'
k = 3 # if using tanh, k is not used 

def u(x):
    return AA *  np.sin(aa * x)

# def d2u_dx2(x):
#     return -AA*(aa*aa) * np.cos(aa*(x+0.05)) 

def F(points, lamb):
    # return(- d2u_dx2(points)  + lamb * u(points)) # change 
    return u(points)

if __name__ == '__main__':
    lamb = 4
    PoU_nums = 5 
    activation = 'tanh'
    k = 2 # if using tanh, k is not used 
    num_trials = 2 
    err_list_Rs = [] 
    R_m_list = [0.75, 1.5,3, 6,9,12] 
    for R_m in R_m_list: # 0.5,1,2 
        print() 
        J_n = 50 # the number of basis functions per PoU region
        Q = 50 # the number of collocation pointss per PoU regio
        RFM_Error = np.zeros([PoU_nums,3])
        err_list = np.zeros([PoU_nums,num_trials])
        for i in range(PoU_nums): # the number of PoU regions
            for trial in range(num_trials): 
                M_p =  2 * (2**i)
                RFM_Error[i,0] = int(M_p * J_n)
                models,w, RFM_Error[i,1], RFM_Error[i,2] = main(M_p,J_n,Q,lamb,activation, k, print_res = False)
                err_list[i,trial] = RFM_Error[i,1] 
        print(err_list.mean(axis = 1)) 
        err_list_Rs.append(err_list.mean(axis = 1)) 
        
    num_diff_features_list = [J_n * 2 * (2**i) for i in range(PoU_nums)]

    print(output_results_Rs_latex(err_list_Rs, R_m_list, num_diff_features_list))

/var/folders/45/p1_q2drx75983lk7ph_1gx6mhpr5_w/T/ipykernel_8943/4274200674.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return(math.sqrt((X_max-X_min)*sum(epsilon*epsilon)/len(epsilon)))


[2.09891673e+00 2.02192010e+00 1.66708332e+00 5.28373156e-03
 1.42143079e-07]

[3.42539594e+00 2.17954106e+00 4.10631280e-01 2.04120776e-06
 1.00798217e-09]

[6.26534321e+01 3.75963794e+01 1.52611467e-02 8.71819286e-07
 1.82261579e-09]

[2.74173050e+03 4.84354711e+02 1.46793415e+01 5.72586449e-02
 2.14426941e-05]

[7.19568541e+03 2.14198049e+05 2.79347189e+01 9.50130803e+00
 8.10264118e-03]

[7.94176855e+03 7.28960623e+03 2.64595224e+03 1.70933622e+01
 6.46816644e+00]
\begin{tabular}{c|cccccc}
\#Features $\backslash$ R & 0.75 & 1.5 & 3 & 6 & 9 & 12 \\
\hline
100 & 2.10e+00 & 3.43e+00 & 6.27e+01 & 2.74e+03 & 7.20e+03 & 7.94e+03 \\
200 & 2.02e+00 & 2.18e+00 & 3.76e+01 & 4.84e+02 & 2.14e+05 & 7.29e+03 \\
400 & 1.67e+00 & 4.11e-01 & 1.53e-02 & 1.47e+01 & 2.79e+01 & 2.65e+03 \\
800 & 5.28e-03 & 2.04e-06 & 8.72e-07 & 5.73e-02 & 9.50e+00 & 1.71e+01 \\
1600 & 1.42e-07 & 1.01e-09 & 1.82e-09 & 2.14e-05 & 8.10e-03 & 6.47e+00 \\
\end{tabular}


In [ ]:

## L^2 minimization 

# analytical solution parameters
activation = 'tanh'
k = 3 # if using tanh, k is not used 

def u(x):
    sigma = 0.15 
    m = 8
    return np.exp(-x**2/(2*sigma**2)) * np.cos(2*np.pi*m*x) # Gabor function

# def d2u_dx2(x):
#     return -AA*(aa*aa) * np.cos(aa*(x+0.05)) 

def F(points, lamb):
    # return(- d2u_dx2(points)  + lamb * u(points)) # change 
    return u(points)

if __name__ == '__main__':
    lamb = 4
    PoU_nums = 5 
    activation = 'tanh'
    k = 2 # if using tanh, k is not used 
    num_trials = 2 
    err_list_Rs = [] 
    R_m_list = [0.75, 1.5,3, 6,9,12] 
    for R_m in R_m_list: # 0.5,1,2 
        print() 
        J_n = 50 # the number of basis functions per PoU region
        Q = 50 # the number of collocation pointss per PoU regio
        RFM_Error = np.zeros([PoU_nums,3])
        err_list = np.zeros([PoU_nums,num_trials])
        for i in range(PoU_nums): # the number of PoU regions
            for trial in range(num_trials): 
                M_p =  2 * (2**i)
                RFM_Error[i,0] = int(M_p * J_n)
                models,w, RFM_Error[i,1], RFM_Error[i,2] = main(M_p,J_n,Q,lamb,activation, k, print_res = False)
                err_list[i,trial] = RFM_Error[i,1] 
        print(err_list.mean(axis = 1)) 
        err_list_Rs.append(err_list.mean(axis = 1)) 
        
    num_diff_features_list = [J_n * 2 * (2**i) for i in range(PoU_nums)]

    print(output_results_Rs_latex(err_list_Rs, R_m_list, num_diff_features_list))

/var/folders/45/p1_q2drx75983lk7ph_1gx6mhpr5_w/T/ipykernel_8943/4274200674.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return(math.sqrt((X_max-X_min)*sum(epsilon*epsilon)/len(epsilon)))


[3.18555448e-01 2.46250763e-01 1.57385383e-01 2.30853781e-03
 6.33541827e-08]

[2.60151567e-01 1.88638326e-01 4.01974033e-02 1.56814977e-06
 4.59177110e-11]

[1.01529786e+00 7.31427545e-01 1.76367778e-03 2.08885244e-07
 2.41842354e-11]

[3.26530390e+01 3.34776342e+00 1.59456819e-02 2.75167140e-04
 3.57407068e-07]

[3.22867993e+00 2.84716356e+02 8.29732141e+00 6.58169403e-03
 2.80154500e-04]

[ 2.31729873 81.35188448  2.27919558  0.35572618  0.23451578]
\begin{tabular}{c|cccccc}
\#Features $\backslash$ R & 0.75 & 1.5 & 3 & 6 & 9 & 12 \\
\hline
100 & 3.19e-01 & 2.60e-01 & 1.02e+00 & 3.27e+01 & 3.23e+00 & 2.32e+00 \\
200 & 2.46e-01 & 1.89e-01 & 7.31e-01 & 3.35e+00 & 2.85e+02 & 8.14e+01 \\
400 & 1.57e-01 & 4.02e-02 & 1.76e-03 & 1.59e-02 & 8.30e+00 & 2.28e+00 \\
800 & 2.31e-03 & 1.57e-06 & 2.09e-07 & 2.75e-04 & 6.58e-03 & 3.56e-01 \\
1600 & 6.34e-08 & 4.59e-11 & 2.42e-11 & 3.57e-07 & 2.80e-04 & 2.35e-01 \\
\end{tabular}


### without partition of unity  

In [74]:
# analytical solution parameters
AA = 1
aa = 1.0*np.pi

lamb = 4
activation = 'tanh'
k = 3 # if using tanh, k is not used 

def u(x):
    return AA * np.sin(aa * (x )) 


def F(points, lamb):
    # return(- d2u_dx2(points)  + lamb * u(points)) # change 
    return u(points)

if __name__ == '__main__':
    lamb = 4
     
    J = 50 # the number of basis functions per PoU region
    Q = 50 # the number of collocation pointss per PoU region
    print_res = False  
    num_diff_features = 5 
    num_trials = 5
    RFM_Error = np.zeros([num_diff_features,3]) 
    err_list =  np.zeros([num_diff_features,num_trials])
    activation = 'tanh'
    relu_k = 3 
    err_list_Rs = [] 
    R_m_list = [1,3,6,12,24,36,48]
    for R_m in R_m_list: 
        print("Using R_m = ", R_m)
        print("========================================") 
        for i in range(num_diff_features): # the number of PoU regions
            for trial in range(num_trials):
                M_p = 1   
                J_n = J * (2**(i+1))
                Q = J_n 
                RFM_Error[i,0] = int(M_p * J_n)
                models,w,RFM_Error[i,1], RFM_Error[i,2] = main(M_p,J_n,Q,lamb,activation=activation, k = relu_k,print_res= print_res)
                err_list[i,trial] = RFM_Error[i,1] 
                
        print(err_list.mean(axis=1)) 
        err_list_Rs.append(err_list.mean(axis = 1))

    num_diff_features_list = [J * 2 * (2**i) for i in range(num_diff_features)]
    print(output_results_Rs_latex(err_list_Rs, R_m_list, num_diff_features_list))


Using R_m =  1


/var/folders/45/p1_q2drx75983lk7ph_1gx6mhpr5_w/T/ipykernel_8943/4274200674.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return(math.sqrt((X_max-X_min)*sum(epsilon*epsilon)/len(epsilon)))


[2.19288323e-06 3.73766406e-06 1.86551818e-05 5.55334282e-06
 1.30438783e-06]
Using R_m =  3
[8.64539815e-10 1.03927249e-10 6.72703870e-11 4.68393519e-11
 4.58101607e-11]
Using R_m =  6
[1.33148424e-08 1.67232616e-11 7.29738322e-13 5.84086810e-13
 5.47411417e-13]
Using R_m =  12
[1.81003425e-04 3.65652142e-10 7.59369302e-13 1.46694860e-13
 1.20629569e-13]
Using R_m =  24
[2.53010885e+00 3.21114421e-04 1.37343666e-11 1.42120684e-13
 5.14005669e-14]
Using R_m =  36
[1.42869705e+03 1.59244820e-02 2.85099485e-07 3.56644314e-13
 7.45241787e-14]
Using R_m =  48
[1.47490259e+02 2.91035883e+00 4.65400943e-05 1.74736769e-12
 9.18683037e-14]
\begin{tabular}{c|ccccccc}
\#Features $\backslash$ R & 1 & 3 & 6 & 12 & 24 & 36 & 48 \\
\hline
100 & 2.19e-06 & 8.65e-10 & 1.33e-08 & 1.81e-04 & 2.53e+00 & 1.43e+03 & 1.47e+02 \\
200 & 3.74e-06 & 1.04e-10 & 1.67e-11 & 3.66e-10 & 3.21e-04 & 1.59e-02 & 2.91e+00 \\
400 & 1.87e-05 & 6.73e-11 & 7.30e-13 & 7.59e-13 & 1.37e-11 & 2.85e-07 & 4.65e-05 \\
800 & 5.55e-0

In [75]:
# analytical solution parameters
AA = 1
aa = 16.0*np.pi

lamb = 4
activation = 'tanh'
k = 3 # if using tanh, k is not used 

def u(x):
    return AA * np.sin(aa * (x )) 


def F(points, lamb):
    # return(- d2u_dx2(points)  + lamb * u(points)) # change 
    return u(points)

if __name__ == '__main__':
    lamb = 4
     
    J = 50 # the number of basis functions per PoU region
    Q = 50 # the number of collocation pointss per PoU region
    print_res = False  
    num_diff_features = 5 
    num_trials = 5   
    RFM_Error = np.zeros([num_diff_features,3]) 
    err_list =  np.zeros([num_diff_features,num_trials])
    activation = 'tanh'
    relu_k = 3 
    err_list_Rs = [] 
    R_m_list = [1,3,6,12,24,36,48]
    for R_m in R_m_list: 
        print("Using R_m = ", R_m)
        print("========================================") 
        for i in range(num_diff_features): # the number of PoU regions
            for trial in range(num_trials):
                M_p = 1   
                J_n = J * (2**(i+1))
                Q = J_n 
                RFM_Error[i,0] = int(M_p * J_n)
                models,w,RFM_Error[i,1], RFM_Error[i,2] = main(M_p,J_n,Q,lamb,activation=activation, k = relu_k,print_res= print_res)
                err_list[i,trial] = RFM_Error[i,1] 
                
        print(err_list.mean(axis=1)) 
        err_list_Rs.append(err_list.mean(axis = 1))

    num_diff_features_list = [J * 2 * (2**i) for i in range(num_diff_features)]
    print(output_results_Rs_latex(err_list_Rs, R_m_list, num_diff_features_list))


Using R_m =  1


/var/folders/45/p1_q2drx75983lk7ph_1gx6mhpr5_w/T/ipykernel_8943/4274200674.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return(math.sqrt((X_max-X_min)*sum(epsilon*epsilon)/len(epsilon)))


[2.00221585 2.75703552 2.59379588 2.22735999 2.03632219]
Using R_m =  3
[2.94022422 2.01915621 1.98951946 1.99128592 1.9856831 ]
Using R_m =  6
[54.45384222  2.0609959   1.97405707  1.96150885  1.96003506]
Using R_m =  12
[1.94067138e+05 1.43940625e+01 1.63286682e+00 4.25918104e-01
 1.38490415e-01]
Using R_m =  24
[3.50941404e+04 2.30198812e+02 1.33641983e-01 8.59512132e-05
 8.62145157e-07]
Using R_m =  36
[2.80173472e+05 9.88310753e+03 2.08041229e+00 9.63651777e-06
 2.24038265e-08]
Using R_m =  48
[6.48894433e+03 6.70792000e+02 5.94957303e+01 1.07166764e-04
 8.45194769e-09]
\begin{tabular}{c|ccccccc}
\#Features $\backslash$ R & 1 & 3 & 6 & 12 & 24 & 36 & 48 \\
\hline
100 & 2.00e+00 & 2.94e+00 & 5.45e+01 & 1.94e+05 & 3.51e+04 & 2.80e+05 & 6.49e+03 \\
200 & 2.76e+00 & 2.02e+00 & 2.06e+00 & 1.44e+01 & 2.30e+02 & 9.88e+03 & 6.71e+02 \\
400 & 2.59e+00 & 1.99e+00 & 1.97e+00 & 1.63e+00 & 1.34e-01 & 2.08e+00 & 5.95e+01 \\
800 & 2.23e+00 & 1.99e+00 & 1.96e+00 & 4.26e-01 & 8.60e-05 & 9.64e-06 &

In [77]:
## L^2 minimization 

# analytical solution parameters
activation = 'tanh'
k = 3 # if using tanh, k is not used 

def u(x):
    sigma = 0.15 
    m = 8
    return np.exp(-x**2/(2*sigma**2)) * np.cos(2*np.pi*m*x) # Gabor function

# def d2u_dx2(x):
#     return -AA*(aa*aa) * np.cos(aa*(x+0.05)) 

def F(points, lamb):
    # return(- d2u_dx2(points)  + lamb * u(points)) # change 
    return u(points)

if __name__ == '__main__':
    lamb = 4
     
    J = 50 # the number of basis functions per PoU region
    Q = 50 # the number of collocation pointss per PoU region
    print_res = False  
    num_diff_features = 5 
    num_trials = 5  
    RFM_Error = np.zeros([num_diff_features,3]) 
    err_list =  np.zeros([num_diff_features,num_trials])
    activation = 'tanh'
    relu_k = 3 
    err_list_Rs = [] 
    R_m_list = [1,3,6,12,24,36,48]
    for R_m in R_m_list: 
        print("Using R_m = ", R_m)
        print("========================================") 
        for i in range(num_diff_features): # the number of PoU regions
            for trial in range(num_trials):
                M_p = 1   
                J_n = J * (2**(i+1))
                Q = J_n 
                RFM_Error[i,0] = int(M_p * J_n)
                models,w,RFM_Error[i,1], RFM_Error[i,2] = main(M_p,J_n,Q,lamb,activation=activation, k = relu_k,print_res= print_res)
                err_list[i,trial] = RFM_Error[i,1] 
                
        print(err_list.mean(axis=1)) 
        err_list_Rs.append(err_list.mean(axis = 1))

    num_diff_features_list = [J * 2 * (2**i) for i in range(num_diff_features)]
    print(output_results_Rs_latex(err_list_Rs, R_m_list, num_diff_features_list))


Using R_m =  1


/var/folders/45/p1_q2drx75983lk7ph_1gx6mhpr5_w/T/ipykernel_8943/4274200674.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return(math.sqrt((X_max-X_min)*sum(epsilon*epsilon)/len(epsilon)))


[0.25526821 0.47506743 0.61275364 0.7141818  0.33189985]
Using R_m =  3
[0.2770526  0.22291958 0.20440466 0.21019223 0.198001  ]
Using R_m =  6
[0.84782758 0.28559597 0.14057964 0.12201523 0.11398497]
Using R_m =  12
[2.03539362 1.69161033 0.03999235 0.01016478 0.00468963]
Using R_m =  24
[2.10843811e+00 6.21691505e+00 1.21016239e-03 8.17675253e-06
 3.73140540e-07]
Using R_m =  36
[2.91434707e+02 2.07042584e+01 2.13723418e-02 7.34557362e-08
 4.98540983e-09]
Using R_m =  48
[1.80091238e+00 4.53435476e+00 5.33347077e-02 1.56477932e-07
 8.62251415e-10]
\begin{tabular}{c|ccccccc}
\#Features $\backslash$ R & 1 & 3 & 6 & 12 & 24 & 36 & 48 \\
\hline
100 & 2.55e-01 & 2.77e-01 & 8.48e-01 & 2.04e+00 & 2.11e+00 & 2.91e+02 & 1.80e+00 \\
200 & 4.75e-01 & 2.23e-01 & 2.86e-01 & 1.69e+00 & 6.22e+00 & 2.07e+01 & 4.53e+00 \\
400 & 6.13e-01 & 2.04e-01 & 1.41e-01 & 4.00e-02 & 1.21e-03 & 2.14e-02 & 5.33e-02 \\
800 & 7.14e-01 & 2.10e-01 & 1.22e-01 & 1.02e-02 & 8.18e-06 & 7.35e-08 & 1.56e-07 \\
1600 & 3.32e-